In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import confusion_matrix

sys.path.insert(0, '..')  # dataset.py lives one level up, shared across experiment types
import dataset
import xgb_train

from investalyze.analysis import encodings
from investalyze.ingest import storage

plt.rcParams['figure.dpi'] = 130

In [ ]:
TICKERS = 50  # list[str] of specific tickers, int for a random sample of that many, or None for all
UNIVERSE = None  # name of a universe saved by the ticker selector app (data/universes/<name>.csv); overrides TICKERS when set
EXCLUDE_TICKERS = [
    'LITX',
    'PCG-PD',
    'LIFE',
    'AKRE',
    'BOBS',
    'BKMI',
    'QBTZ',
    'XWIN',
    'SHAZ',
    'OPENL',
    'HOYY',
    'JBS',
    'NSIT',
]  # tickers to always leave out, regardless of TICKERS or UNIVERSE
SEED = 0  # used when TICKERS is an int, and when VALID_METHOD is 'random'
WINDOW_LENGTH = 30
STRIDE = 5

VALID_FRAC = 0.3
VALID_METHOD = 'random'  # 'recent' = time-based (no leakage) / 'random' = random per-window
TEST_N = 2  # always recent: the last N windows of each ticker

ENCODER = encodings.RebaseTo1  # swap by hand: RebaseTo100 / RebaseTo1 / encodings.zscore / encodings.minmax

N_ESTIMATORS = 500  # boosting rounds (upper bound; early stopping picks the best)
MAX_DEPTH = 6
LEARNING_RATE = 0.1
EARLY_STOPPING_ROUNDS = 20  # stop if valid mlogloss hasn't improved in this many rounds
N_JOBS = -1

In [ ]:
DATA_ROOT = Path('../../data')

con = storage.connect(DATA_ROOT, read_only=True)
if UNIVERSE is not None:
    tickers = [t for t in dataset.load_universe(UNIVERSE, DATA_ROOT) if t not in EXCLUDE_TICKERS]
elif isinstance(TICKERS, int):
    tickers = dataset.sample_tickers(con, TICKERS, seed=SEED, exclude=EXCLUDE_TICKERS)
else:
    tickers = TICKERS
series = dataset.get_ohlcv_series(con, tickers, exclude=EXCLUDE_TICKERS)
con.close()
print('tickers used', tickers if tickers is not None else 'ALL')

channels, meta = dataset.build_windows(series, window_length=WINDOW_LENGTH, stride=STRIDE)
print('windows', meta.shape[0], 'across', meta['Ticker'].nunique(), 'tickers')

In [ ]:
train_mask, valid_mask, test_mask = dataset.split_windows(meta, valid_frac=VALID_FRAC, valid_method=VALID_METHOD, test_n=TEST_N, seed=SEED)

labels, tickers_sorted = dataset.encode_labels(meta, train_mask)
n_classes = len(tickers_sorted)

print('train windows', int(train_mask.sum()), 'valid windows', int(valid_mask.sum()), 'test windows', int(test_mask.sum()))
pd.Series(labels[train_mask]).value_counts().sort_index()

In [ ]:
X = dataset.encode_windows(channels, ENCODER)
X_flat = dataset.flatten_windows(X)
X_train, y_train = X_flat[train_mask], labels[train_mask]
X_valid, y_valid = X_flat[valid_mask], labels[valid_mask]
X_test, y_test = X_flat[test_mask], labels[test_mask]

weights = dataset.class_weights(y_train, n_classes)
sample_weight = weights[y_train]

In [ ]:
clf = xgb.XGBClassifier(
    n_estimators=N_ESTIMATORS,
    max_depth=MAX_DEPTH,
    learning_rate=LEARNING_RATE,
    tree_method='hist',
    early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    eval_metric='mlogloss',
    n_jobs=N_JOBS,
)
clf.fit(X_train, y_train, sample_weight=sample_weight, eval_set=[(X_valid, y_valid)], verbose=False)
print('best iteration', clf.best_iteration, '/', N_ESTIMATORS)

In [ ]:
valid_logloss = clf.evals_result()['validation_0']['mlogloss']
plt.figure(figsize=(8, 4))
plt.plot(valid_logloss)
plt.axvline(clf.best_iteration, color='gray', linestyle='--', label='best iteration')
plt.xlabel('boosting round')
plt.ylabel('valid mlogloss')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
valid_preds = clf.predict(X_valid)
valid_true = y_valid
valid_macro_acc = xgb_train.macro_accuracy(valid_preds, valid_true, n_classes)
print(f'valid accuracy={(valid_preds == valid_true).mean():.3f}  macro-accuracy={valid_macro_acc:.3f}')

cm_valid = confusion_matrix(valid_true, valid_preds, labels=range(n_classes))
row_sums = cm_valid.sum(axis=1, keepdims=True)
cm_valid_norm = np.divide(cm_valid, row_sums, out=np.zeros_like(cm_valid, dtype=float), where=row_sums != 0)

fig, ax = plt.subplots(figsize=(6, 6))
im = ax.imshow(cm_valid_norm, cmap='viridis', norm=mcolors.PowerNorm(gamma=0.4, vmin=0, vmax=1), interpolation='nearest')
if n_classes <= 40:
    ax.set_xticks(range(n_classes))
    ax.set_xticklabels(tickers_sorted, rotation=90)
    ax.set_yticks(range(n_classes))
    ax.set_yticklabels(tickers_sorted)
else:
    ax.set_xticks([])
    ax.set_yticks([])
ax.set_xlabel('predicted')
ax.set_ylabel('true')
ax.set_title('valid set (row-normalized, gamma=0.4)')
plt.colorbar(im, label='fraction of true class')
plt.tight_layout()
plt.show()

In [ ]:
cm_valid_offdiag = cm_valid.copy()
np.fill_diagonal(cm_valid_offdiag, 0)
i_idx, j_idx = np.nonzero(cm_valid_offdiag)
counts = cm_valid_offdiag[i_idx, j_idx]
order = np.argsort(-counts)[:15]
valid_pairs = [(tickers_sorted[i], tickers_sorted[j], int(c)) for i, j, c in zip(i_idx[order], j_idx[order], counts[order])]
valid_pairs

In [ ]:
preds, y_true = clf.predict(X_test), y_test
test_acc = (preds == y_true).mean()
test_macro_acc = xgb_train.macro_accuracy(preds, y_true, n_classes)
print(f'test accuracy={test_acc:.3f}  macro-accuracy={test_macro_acc:.3f}')

cm = confusion_matrix(y_true, preds, labels=range(n_classes))
row_sums = cm.sum(axis=1, keepdims=True)
cm_norm = np.divide(cm, row_sums, out=np.zeros_like(cm, dtype=float), where=row_sums != 0)

fig, ax = plt.subplots(figsize=(6, 6))
im = ax.imshow(cm_norm, cmap='viridis', norm=mcolors.PowerNorm(gamma=0.4, vmin=0, vmax=1), interpolation='nearest')
if n_classes <= 40:
    ax.set_xticks(range(n_classes))
    ax.set_xticklabels(tickers_sorted, rotation=90)
    ax.set_yticks(range(n_classes))
    ax.set_yticklabels(tickers_sorted)
else:
    ax.set_xticks([])
    ax.set_yticks([])
ax.set_xlabel('predicted')
ax.set_ylabel('true')
ax.set_title('test set (row-normalized, gamma=0.4)')
plt.colorbar(im, label='fraction of true class')
plt.tight_layout()
plt.show()

In [ ]:
cm_offdiag = cm.copy()
np.fill_diagonal(cm_offdiag, 0)
i_idx, j_idx = np.nonzero(cm_offdiag)
counts = cm_offdiag[i_idx, j_idx]
order = np.argsort(-counts)[:15]
pairs = [(tickers_sorted[i], tickers_sorted[j], int(c)) for i, j, c in zip(i_idx[order], j_idx[order], counts[order])]
pairs

In [ ]:
def worst_predicted(cm_offdiag, top=10):
    fp_counts = cm_offdiag.sum(axis=0)  # column sums: how often each ticker is wrongly predicted
    n_sources = (cm_offdiag > 0).sum(axis=0)  # distinct true tickers wrongly predicted as this one
    order = np.argsort(-fp_counts)[:top]
    return order, fp_counts, n_sources


for name, offdiag in [('valid', cm_valid_offdiag), ('test', cm_offdiag)]:
    order, fp_counts, n_sources = worst_predicted(offdiag)
    print(f'{name}:')
    for idx in order:
        pct_sources = n_sources[idx] / n_classes * 100
        print(
            f'  {tickers_sorted[idx]:>8}  wrongly predicted {int(fp_counts[idx])} times, '
            f'from {int(n_sources[idx])} different tickers ({pct_sources:.1f}% of all tickers)'
        )
    print()